# 🫀 Notebook 2 — ECG Signal Preprocessing
### Project: Explainable Lightweight Edge-AI Framework for Early Cardiac Risk Prediction using Attention-Based Deep Learning on ECG Signals
**Week 1 | Task 2: Signal Cleaning, Segmentation & Feature Preparation**

---
**Preprocessing Pipeline:**
1. Noise Removal (Bandpass Filter)
2. Baseline Wander Correction
3. Signal Normalization
4. R-Peak Detection
5. Heartbeat Segmentation (Windows)
6. Cardiac Risk Label Assignment
7. Dataset Saving for Deep Learning
---

## 📦 Cell 1 — Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk

from scipy.signal import butter, filtfilt, iirnotch
from scipy import signal as sp_signal

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Style ─────────────────────────────────────────────────────────
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['axes.edgecolor']   = '#444'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#333'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5

# ── Create output folder ──────────────────────────────────────────
os.makedirs('./outputs', exist_ok=True)
os.makedirs('./data/processed', exist_ok=True)

print('✅ Libraries imported and folders created!')

---
## ⚙️ Cell 2 — Configuration & Constants

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────
MIT_BIH_PATH = './data/mit-bih-arrhythmia-database-1.0.0'  # Update if needed

# ── Signal Parameters ─────────────────────────────────────────────
SAMPLING_RATE  = 360          # MIT-BIH sampling frequency (Hz)
WINDOW_SIZE    = 180          # Samples per heartbeat window (~0.5 sec at 360 Hz)
BEFORE_R       = 90           # Samples before R-peak
AFTER_R        = 90           # Samples after R-peak

# ── Filter Parameters ─────────────────────────────────────────────
LOWCUT         = 0.5          # Hz — removes baseline wander
HIGHCUT        = 50.0         # Hz — removes high-freq noise
NOTCH_FREQ     = 60.0         # Hz — removes powerline interference (50 Hz for India/EU)
FILTER_ORDER   = 4

# ── AAMI Beat Categories → Cardiac Risk Labels ────────────────────
# Beat type → Risk Category mapping (clinical rule-based)
BEAT_TO_RISK = {
    # Low Risk — Normal/Benign
    'N' : 0,   # Normal
    'L' : 0,   # Left Bundle Branch Block (mild)
    'R' : 0,   # Right Bundle Branch Block (mild)
    'e' : 0,   # Atrial Escape
    '/' : 0,   # Paced Beat

    # Moderate Risk — Supraventricular
    'A' : 1,   # Atrial Premature Beat
    'a' : 1,   # Aberrated Atrial Premature
    'J' : 1,   # Nodal Premature Beat
    'S' : 1,   # Supraventricular Premature
    'j' : 1,   # Nodal Escape

    # High Risk — Ventricular
    'V' : 2,   # Premature Ventricular Contraction
    'E' : 2,   # Ventricular Escape
    'f' : 2,   # Fusion of Paced and Normal
    'F' : 2,   # Fusion of Ventricular and Normal

    # Critical Risk — Life-Threatening
    '[' : 3,   # Ventricular Flutter start
    '!' : 3,   # Ventricular Flutter Wave
    ']' : 3,   # Ventricular Flutter end
    'Q' : 3,   # Unclassifiable (treat as critical)
}

RISK_NAMES = {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk', 3: 'Critical Risk'}
RISK_COLORS = {0: '#00ff88', 1: '#ffd32a', 2: '#ff6b35', 3: '#ff4757'}

# ── MIT-BIH records ───────────────────────────────────────────────
MIT_BIH_RECORDS = [
    '100','101','102','103','104','105','106','107',
    '108','109','111','112','113','114','115','116',
    '117','118','119','121','122','123','124','200',
    '201','202','203','205','207','208','209','210',
    '212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

print('✅ Configuration set!')
print(f'   Window Size   : {WINDOW_SIZE} samples ({WINDOW_SIZE/SAMPLING_RATE*1000:.0f} ms)')
print(f'   Before R-peak : {BEFORE_R} samples')
print(f'   After  R-peak : {AFTER_R} samples')
print(f'   Filter        : {LOWCUT}–{HIGHCUT} Hz bandpass  +  {NOTCH_FREQ} Hz notch')
print(f'\n   Risk Label Map:')
for k, v in RISK_NAMES.items():
    beats = [b for b, r in BEAT_TO_RISK.items() if r == k]
    print(f'     {k} → {v}  {beats}')

---
## 🔧 Cell 3 — Preprocessing Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  FUNCTION 1: Bandpass Filter (removes noise & baseline wander)
# ═══════════════════════════════════════════════════════════════════
def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    """
    Apply a Butterworth bandpass filter to remove:
    - Low-freq baseline wander (< 0.5 Hz)
    - High-freq muscle noise (> 50 Hz)
    """
    nyquist = fs / 2.0
    low     = lowcut  / nyquist
    high    = highcut / nyquist
    b, a    = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


# ═══════════════════════════════════════════════════════════════════
#  FUNCTION 2: Notch Filter (removes powerline interference)
# ═══════════════════════════════════════════════════════════════════
def notch_filter(signal, notch_freq, fs, quality_factor=30):
    """
    Remove powerline interference (50 Hz in India/EU, 60 Hz in US)
    """
    nyquist = fs / 2.0
    freq    = notch_freq / nyquist
    b, a    = iirnotch(freq, quality_factor)
    return filtfilt(b, a, signal)


# ═══════════════════════════════════════════════════════════════════
#  FUNCTION 3: Normalize Signal (zero-mean, unit-variance)
# ═══════════════════════════════════════════════════════════════════
def normalize_signal(signal):
    """
    Z-score normalization: (x - mean) / std
    Ensures consistent amplitude range across different patients/leads.
    """
    mean = np.mean(signal)
    std  = np.std(signal)
    if std < 1e-8:    # avoid division by zero for flat signals
        return signal - mean
    return (signal - mean) / std


# ═══════════════════════════════════════════════════════════════════
#  FUNCTION 4: Full ECG Preprocessing Pipeline
# ═══════════════════════════════════════════════════════════════════
def preprocess_ecg(raw_signal, fs=360):
    """
    Complete preprocessing pipeline:
    1. Bandpass filter (0.5–50 Hz)
    2. Notch filter (60 Hz)
    3. Z-score normalization
    Returns: cleaned signal array
    """
    # Step 1: Bandpass filter
    filtered = bandpass_filter(raw_signal, LOWCUT, HIGHCUT, fs, FILTER_ORDER)
    # Step 2: Notch filter
    filtered = notch_filter(filtered, NOTCH_FREQ, fs)
    # Step 3: Normalize
    normalized = normalize_signal(filtered)
    return normalized


# ═══════════════════════════════════════════════════════════════════
#  FUNCTION 5: R-Peak Detection
# ═══════════════════════════════════════════════════════════════════
def detect_r_peaks(signal, fs=360):
    """
    Detect R-peaks using NeuroKit2's Pan-Tompkins implementation.
    Returns: array of R-peak sample indices
    """
    try:
        ecg_processed, info = nk.ecg_process(signal, sampling_rate=fs)
        r_peaks = info['ECG_R_Peaks']
        return r_peaks
    except Exception:
        return np.array([])


# ═══════════════════════════════════════════════════════════════════
#  FUNCTION 6: Segment heartbeats around R-peaks
# ═══════════════════════════════════════════════════════════════════
def segment_heartbeats(signal, r_peaks, before=90, after=90):
    """
    Extract fixed-length windows centered on each R-peak.
    Skips peaks too close to start or end of signal.
    Returns: list of (window, r_peak_position) tuples
    """
    segments = []
    for r in r_peaks:
        start = r - before
        end   = r + after
        if start >= 0 and end <= len(signal):
            window = signal[start:end]
            segments.append(window)
    return segments


print('✅ All preprocessing functions defined!')
print('   Functions available:')
print('   1. bandpass_filter()     — remove noise & baseline wander')
print('   2. notch_filter()        — remove powerline interference')
print('   3. normalize_signal()    — z-score normalization')
print('   4. preprocess_ecg()      — full pipeline (1+2+3)')
print('   5. detect_r_peaks()      — Pan-Tompkins R-peak detection')
print('   6. segment_heartbeats()  — extract fixed windows')

---
## 🔬 Cell 4 — Apply Preprocessing on One Record (Visual Demo)

In [ ]:
# ── Load record 100 ───────────────────────────────────────────────
DEMO_RECORD = '100'
rec_path    = os.path.join(MIT_BIH_PATH, DEMO_RECORD)

record      = wfdb.rdrecord(rec_path)
annotation  = wfdb.rdann(rec_path, 'atr')

raw_ecg     = record.p_signal[:, 0].astype(np.float64)  # Channel 1 (MLII)
fs          = record.fs

# ── Apply preprocessing step-by-step ─────────────────────────────
step1_bp    = bandpass_filter(raw_ecg, LOWCUT, HIGHCUT, fs)
step2_notch = notch_filter(step1_bp, NOTCH_FREQ, fs)
step3_norm  = normalize_signal(step2_notch)

print(f'✅ Preprocessing applied on Record {DEMO_RECORD}')
print(f'\n   Raw Signal     → min={raw_ecg.min():.4f}  max={raw_ecg.max():.4f}  std={raw_ecg.std():.4f}')
print(f'   After Bandpass → min={step1_bp.min():.4f}  max={step1_bp.max():.4f}  std={step1_bp.std():.4f}')
print(f'   After Notch    → min={step2_notch.min():.4f}  max={step2_notch.max():.4f}  std={step2_notch.std():.4f}')
print(f'   After Norm     → min={step3_norm.min():.4f}  max={step3_norm.max():.4f}  std={step3_norm.std():.4f}')

---
## 📊 Cell 5 — Before vs After Preprocessing Visualization

In [ ]:
# ── Plot 5-second comparison: raw vs each preprocessing step ─────
PLOT_SEC  = 5
n_samples = PLOT_SEC * fs
t_axis    = np.arange(n_samples) / fs

signals = {
    'Raw Signal (Original)'             : raw_ecg[:n_samples],
    'After Bandpass Filter (0.5–50 Hz)' : step1_bp[:n_samples],
    'After Notch Filter (60 Hz removed)': step2_notch[:n_samples],
    'After Z-score Normalization'        : step3_norm[:n_samples],
}
colors = ['#aaaaaa', '#00d4ff', '#ffd32a', '#00ff88']

fig, axes = plt.subplots(4, 1, figsize=(18, 10), sharex=True)
for ax, (title, sig), color in zip(axes, signals.items(), colors):
    ax.plot(t_axis, sig, color=color, linewidth=0.9)
    ax.set_title(title, fontsize=11, color='white')
    ax.set_ylabel('Amplitude', fontsize=9)
    ax.grid(True)

axes[-1].set_xlabel('Time (seconds)', fontsize=11)
fig.suptitle(f'ECG Preprocessing Pipeline — Record {DEMO_RECORD}', 
             fontsize=14, color='white', y=1.01)
plt.tight_layout()
plt.savefig('./outputs/fig5_preprocessing_pipeline.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig5_preprocessing_pipeline.png')

---
## 💓 Cell 6 — R-Peak Detection Visualization

In [ ]:
# ── Detect R-peaks on 10-second window ───────────────────────────
DETECT_SEC   = 10
n_detect     = DETECT_SEC * fs
ecg_clean    = step3_norm[:n_detect]
time_detect  = np.arange(n_detect) / fs

r_peaks      = detect_r_peaks(ecg_clean, fs)

print(f'✅ R-peak detection complete!')
print(f'   Signal duration : {DETECT_SEC} seconds')
print(f'   R-peaks detected: {len(r_peaks)}')

# Calculate RR intervals
if len(r_peaks) > 1:
    rr_intervals_ms = np.diff(r_peaks) / fs * 1000   # in milliseconds
    heart_rate_bpm  = 60000 / rr_intervals_ms
    print(f'   Mean RR interval: {rr_intervals_ms.mean():.1f} ms')
    print(f'   Heart Rate      : {heart_rate_bpm.mean():.1f} BPM')

# ── Plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 7))

# Top: ECG with R-peaks marked
axes[0].plot(time_detect, ecg_clean, color='#00d4ff', linewidth=0.9, label='Clean ECG')
if len(r_peaks) > 0:
    axes[0].scatter(r_peaks / fs, ecg_clean[r_peaks],
                    color='#ff4757', s=60, zorder=5, label=f'R-peaks (n={len(r_peaks)})')
axes[0].set_title('Clean ECG with R-Peak Detection', fontsize=12, color='white')
axes[0].set_ylabel('Normalized Amplitude', fontsize=10)
axes[0].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
axes[0].grid(True)

# Bottom: RR interval plot (heart rate variability)
if len(r_peaks) > 1:
    axes[1].plot(range(len(rr_intervals_ms)), rr_intervals_ms,
                 color='#ffd32a', linewidth=1.5, marker='o', markersize=4, label='RR Interval')
    axes[1].axhline(y=rr_intervals_ms.mean(), color='#ff6b9d', linestyle='--',
                    linewidth=1.5, label=f'Mean = {rr_intervals_ms.mean():.1f} ms')
    axes[1].set_title('RR Intervals (Heart Rate Variability)', fontsize=12, color='white')
    axes[1].set_xlabel('Beat Index', fontsize=10)
    axes[1].set_ylabel('RR Interval (ms)', fontsize=10)
    axes[1].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
    axes[1].grid(True)

plt.tight_layout()
plt.savefig('./outputs/fig6_r_peak_detection.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig6_r_peak_detection.png')

---
## ✂️ Cell 7 — Heartbeat Segmentation Visualization

In [ ]:
# ── Segment beats from cleaned full signal ────────────────────────
r_peaks_full = detect_r_peaks(step3_norm, fs)
segments     = segment_heartbeats(step3_norm, r_peaks_full, BEFORE_R, AFTER_R)

print(f'✅ Heartbeat segmentation complete!')
print(f'   Total R-peaks  : {len(r_peaks_full)}')
print(f'   Valid segments : {len(segments)}')
print(f'   Window size    : {WINDOW_SIZE} samples ({WINDOW_SIZE/fs*1000:.0f} ms)')
print(f'   Shape of each  : ({WINDOW_SIZE},)')

if segments:
    arr = np.array(segments)
    print(f'   Full array     : {arr.shape}  (beats × samples)')

# ── Plot 6 sample heartbeat segments ─────────────────────────────
if len(segments) >= 6:
    t_beat = np.arange(WINDOW_SIZE) / fs * 1000   # in ms
    n_plot = 6
    indices = np.linspace(0, len(segments)-1, n_plot, dtype=int)

    fig, axes = plt.subplots(2, 3, figsize=(16, 7))
    axes = axes.flatten()

    for i, (ax, idx) in enumerate(zip(axes, indices)):
        beat = segments[idx]
        ax.plot(t_beat, beat, color='#00d4ff', linewidth=1.2)
        ax.axvline(x=BEFORE_R/fs*1000, color='#ff4757', linestyle='--',
                   linewidth=1, alpha=0.8, label='R-peak')
        ax.fill_between(t_beat, beat, alpha=0.15, color='#00d4ff')
        ax.set_title(f'Heartbeat #{idx}', fontsize=10, color='white')
        ax.set_xlabel('Time (ms)', fontsize=8)
        ax.set_ylabel('Amplitude', fontsize=8)
        ax.grid(True)
        if i == 0:
            ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=8)

    fig.suptitle('Sample Heartbeat Segments (180-sample Windows)', fontsize=13, color='white')
    plt.tight_layout()
    plt.savefig('./outputs/fig7_heartbeat_segments.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('✅ Figure saved → outputs/fig7_heartbeat_segments.png')

---
## 🏥 Cell 8 — Cardiac Risk Label Assignment

In [ ]:
# ── Match annotation labels to R-peaks ────────────────────────────
ann_samples = np.array(annotation.sample)
ann_symbols = np.array(annotation.symbol)

def get_label_for_rpeak(r_peak, ann_samples, ann_symbols, tolerance=15):
    """
    Find the annotation closest to a given R-peak sample position.
    tolerance: max sample distance to accept a match
    """
    dists = np.abs(ann_samples - r_peak)
    idx   = np.argmin(dists)
    if dists[idx] <= tolerance:
        return ann_symbols[idx]
    return None

# ── Assign risk labels to each R-peak ────────────────────────────
beat_labels  = []
risk_labels  = []
valid_peaks  = []

for r in r_peaks_full:
    sym = get_label_for_rpeak(r, ann_samples, ann_symbols)
    if sym is not None and sym in BEAT_TO_RISK:
        beat_labels.append(sym)
        risk_labels.append(BEAT_TO_RISK[sym])
        valid_peaks.append(r)

beat_labels = np.array(beat_labels)
risk_labels = np.array(risk_labels)
valid_peaks = np.array(valid_peaks)

print(f'✅ Risk labels assigned!')
print(f'   Total R-peaks        : {len(r_peaks_full)}')
print(f'   Labeled heartbeats   : {len(risk_labels)}')
print()
for r_id, r_name in RISK_NAMES.items():
    count = np.sum(risk_labels == r_id)
    pct   = 100 * count / len(risk_labels) if len(risk_labels) > 0 else 0
    bar   = '█' * int(pct / 2)
    print(f'   {r_name:15s} (label {r_id}): {count:5d} beats  {pct:5.1f}%  {bar}')

---
## 🏭 Cell 9 — Process ALL MIT-BIH Records & Build Full Dataset

In [ ]:
# ── Full dataset construction loop ────────────────────────────────
# This processes all available MIT-BIH records

all_segments   = []
all_risk_labels = []
all_beat_labels = []
records_ok      = 0
records_fail    = 0

print('🔄 Processing all MIT-BIH records...')
print('-' * 55)

for rec_id in MIT_BIH_RECORDS:
    try:
        rec_path  = os.path.join(MIT_BIH_PATH, rec_id)
        rec       = wfdb.rdrecord(rec_path)
        ann       = wfdb.rdann(rec_path, 'atr')

        raw       = rec.p_signal[:, 0].astype(np.float64)
        fs_rec    = rec.fs

        # 1. Preprocess
        cleaned   = preprocess_ecg(raw, fs_rec)

        # 2. Detect R-peaks
        r_peaks   = detect_r_peaks(cleaned, fs_rec)
        if len(r_peaks) < 10:
            raise ValueError('Too few R-peaks detected')

        # 3. Segment
        a_samples = np.array(ann.sample)
        a_symbols = np.array(ann.symbol)

        for r in r_peaks:
            sym = get_label_for_rpeak(r, a_samples, a_symbols)
            if sym is None or sym not in BEAT_TO_RISK:
                continue
            start = r - BEFORE_R
            end   = r + AFTER_R
            if start < 0 or end > len(cleaned):
                continue
            window = cleaned[start:end]
            all_segments.append(window)
            all_risk_labels.append(BEAT_TO_RISK[sym])
            all_beat_labels.append(sym)

        records_ok += 1
        print(f'   ✅ {rec_id}  → {len(r_peaks)} peaks processed')

    except Exception as e:
        records_fail += 1
        print(f'   ⚠️  {rec_id}  → Skipped: {e}')

print('-' * 55)
print(f'\n✅ Done! {records_ok} records processed, {records_fail} skipped')
print(f'📊 Total segments collected: {len(all_segments)}')

---
## 📊 Cell 10 — Final Dataset Statistics & Risk Class Distribution

In [ ]:
# ── Convert to NumPy arrays ───────────────────────────────────────
X = np.array(all_segments,    dtype=np.float32)    # shape: (N, 180)
y = np.array(all_risk_labels, dtype=np.int32)       # shape: (N,)
b = np.array(all_beat_labels)                       # shape: (N,)

print(f'\n📐 Dataset Shape:')
print(f'   X (segments) : {X.shape}   → {X.shape[0]} beats × {X.shape[1]} samples')
print(f'   y (labels)   : {y.shape}')
print(f'   X dtype      : {X.dtype}')
print(f'   X range      : [{X.min():.4f}, {X.max():.4f}]')

print(f'\n📊 Cardiac Risk Class Distribution:')
print('-' * 50)
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    pct = 100 * count / len(y)
    print(f'   {RISK_NAMES[label]:15s} (label {label}): {count:6d}  ({pct:.1f}%)')
print('-' * 50)
print(f'   {"TOTAL":<15}          : {len(y):6d}  (100%)')

# ── Plot class distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

risk_names_list  = [RISK_NAMES[l]  for l in unique]
risk_colors_list = [RISK_COLORS[l] for l in unique]

# Bar chart
bars = axes[0].bar(risk_names_list, counts, color=risk_colors_list, edgecolor='white', linewidth=0.5)
axes[0].set_title('Cardiac Risk Class Distribution', fontsize=12, color='white')
axes[0].set_ylabel('Number of Beats', fontsize=10)
axes[0].set_xlabel('Risk Category', fontsize=10)
axes[0].grid(axis='y')
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{cnt:,}', ha='center', fontsize=9, color='white')

# Donut
wedge_props = dict(width=0.5, edgecolor='#0f0f0f', linewidth=2)
axes[1].pie(counts, labels=risk_names_list, autopct='%1.1f%%',
            colors=risk_colors_list, wedgeprops=wedge_props,
            textprops={'color': 'white', 'fontsize': 10})
axes[1].set_title('Risk Class Proportion', fontsize=12, color='white')

plt.tight_layout()
plt.savefig('./outputs/fig8_risk_class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig8_risk_class_distribution.png')

---
## 💾 Cell 11 — Save Processed Dataset

In [ ]:
# ── Save processed arrays ─────────────────────────────────────────
SAVE_PATH = './data/processed'
os.makedirs(SAVE_PATH, exist_ok=True)

np.save(os.path.join(SAVE_PATH, 'X_segments.npy'),    X)
np.save(os.path.join(SAVE_PATH, 'y_risk_labels.npy'), y)
np.save(os.path.join(SAVE_PATH, 'beat_symbols.npy'),  b)

# Save metadata CSV
df_meta = pd.DataFrame({
    'beat_symbol' : b,
    'risk_label'  : y,
    'risk_name'   : [RISK_NAMES[l] for l in y]
})
df_meta.to_csv(os.path.join(SAVE_PATH, 'dataset_metadata.csv'), index=False)

print('✅ Dataset saved successfully!')
print(f'\n   📁 Location: {SAVE_PATH}/')
print(f'   📄 X_segments.npy    → {X.shape}  ({X.nbytes/1e6:.2f} MB)')
print(f'   📄 y_risk_labels.npy → {y.shape}')
print(f'   📄 beat_symbols.npy  → {b.shape}')
print(f'   📄 dataset_metadata.csv')

# Verify by reloading
X_check = np.load(os.path.join(SAVE_PATH, 'X_segments.npy'))
y_check = np.load(os.path.join(SAVE_PATH, 'y_risk_labels.npy'))
print(f'\n🔍 Verification reload:')
print(f'   X_check shape : {X_check.shape}  ✅')
print(f'   y_check shape : {y_check.shape}  ✅')

---
## 📊 Cell 12 — Sample Beats per Risk Category

In [ ]:
# ── Show 3 sample beats per risk category ────────────────────────
t_beat = np.arange(WINDOW_SIZE) / SAMPLING_RATE * 1000  # ms

fig, axes = plt.subplots(4, 3, figsize=(16, 12))

for row, (risk_id, risk_name) in enumerate(RISK_NAMES.items()):
    indices = np.where(y == risk_id)[0]
    color   = RISK_COLORS[risk_id]

    if len(indices) >= 3:
        sample_indices = indices[:3]
    else:
        sample_indices = indices

    for col, idx in enumerate(sample_indices[:3]):
        ax = axes[row][col]
        ax.plot(t_beat, X[idx], color=color, linewidth=1.2)
        ax.fill_between(t_beat, X[idx], alpha=0.15, color=color)
        ax.axvline(x=BEFORE_R/SAMPLING_RATE*1000, color='white',
                   linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_title(f'{risk_name} — Beat #{idx}\n[{b[idx]}]', fontsize=9, color=color)
        ax.grid(True)
        ax.set_xlabel('ms', fontsize=7)

    # If fewer than 3 samples
    for col in range(len(sample_indices), 3):
        axes[row][col].axis('off')
        axes[row][col].text(0.5, 0.5, 'No more samples', ha='center',
                             va='center', color='gray', fontsize=10)

fig.suptitle('Sample Heartbeats by Cardiac Risk Category', fontsize=14, color='white', y=1.01)
plt.tight_layout()
plt.savefig('./outputs/fig9_beats_by_risk_category.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig9_beats_by_risk_category.png')

---
## ✅ Cell 13 — Week 1 Summary & Deliverables

In [ ]:
print('=' * 60)
print('  🎉 WEEK 1 COMPLETE — All Deliverables Ready!')
print('=' * 60)

print('\n📁 Notebook 1 — Dataset Loading & Exploration:')
print('   ✅ MIT-BIH dataset loaded & visualized')
print('   ✅ PTB dataset loaded & visualized')
print('   ✅ Beat annotations explored')
print('   ✅ Dataset statistics generated')
print('   ✅ Figures: fig1, fig2, fig3, fig4')

print('\n📁 Notebook 2 — ECG Signal Preprocessing:')
print('   ✅ Bandpass filter applied (0.5–50 Hz)')
print('   ✅ Notch filter applied (60 Hz)')
print('   ✅ Z-score normalization applied')
print('   ✅ R-peak detection (NeuroKit2 Pan-Tompkins)')
print('   ✅ Heartbeat segmentation (180-sample windows)')
print('   ✅ Cardiac risk labels assigned (4 classes)')
print('   ✅ Full dataset saved to data/processed/')
print('   ✅ Figures: fig5, fig6, fig7, fig8, fig9')

print(f'\n📊 Final Dataset:')
print(f'   X shape : {X.shape}')
print(f'   y shape : {y.shape}')
print(f'   Classes : 0=Low Risk  1=Moderate  2=High  3=Critical')

print('\n➡️  Next Step: Notebook 3 — Deep Learning Model Development')
print('=' * 60)

---
## ✅ Week 1 Checklist

| Task | Status |
|------|--------|
| Bandpass filter (0.5–50 Hz) | ✅ Done |
| Notch filter (60 Hz powerline removal) | ✅ Done |
| Z-score normalization | ✅ Done |
| R-peak detection (Pan-Tompkins via NeuroKit2) | ✅ Done |
| 180-sample heartbeat segmentation | ✅ Done |
| Risk labels (Low / Moderate / High / Critical) | ✅ Done |
| Full dataset saved (.npy format) | ✅ Done |
| 9 publication-quality figures generated | ✅ Done |

**Files ready for Week 2:**
- `data/processed/X_segments.npy` — Input features
- `data/processed/y_risk_labels.npy` — Risk labels
- `data/processed/dataset_metadata.csv` — Beat metadata